In [ ]:
import numpy as np
import xarray as xr
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
# from matplotlib.ticker import FormatStrFormatter
from matplotlib.ticker import StrMethodFormatter
# from matplotlib.ticker import ScalarFormatter
# from matplotlib.colors import SymLogNorm

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
def ax_pos_inch_to_absolute(fig_size, ax_pos_inch):
    ax_pos_absolute = []
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])

    return ax_pos_absolute

In [ ]:
def ax_pos_cm_to_absolute(fig_size, ax_pos_cm):
    ax_pos_absolute = []
    ax_pos_inch = [ pos / 2.54 for pos in ax_pos_cm ]
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])
    
    return ax_pos_absolute

In [ ]:
# setting up a custom blue-white-orange color map
from matplotlib.colors import LinearSegmentedColormap

colors = np.array([( 37,  52, 148), ( 44, 127, 184), ( 65, 182, 196), (161, 218, 180), (255, 255, 204),
                   (255, 255, 255),
                   (255, 255, 204), (254, 204,  92), (253, 141,  60), (240,  59,  32), (189,   0,  38)]) / 255.

cmc = LinearSegmentedColormap.from_list('BYWYR', colors, N=101)


In [ ]:
def angle_to_l(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 180. / x[~near_zero]
    return x


In [ ]:
def freq_to_time(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 1. / x[~near_zero]
    return x


In [ ]:
def wavenumber_to_wavelength(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 360. / x[~near_zero]
    return x


In [ ]:
# grid
ntrunc=72
nSampWin = 720
nWindow = 59
spd = 2
frequency = np.fft.fftfreq(nSampWin, 1./spd)
ff = frequency[0:int(nSampWin/2)]
ll = np.arange(0, ntrunc+1, 1)
mm = np.arange(-ntrunc, ntrunc+1, 1)

In [ ]:
# estimated parameters
epsilon_0 = 5.8
lambda_0 = 0.060
tau_0 = 2.3


In [ ]:
# comp psd -- l
def psd_l(Fwflm):

    # compute power
    Pwflm = np.abs(Fwflm[:, :, :, :])**2

    # to get psd need to scale (divide) by frequency spacing = spd/(nSampWin)
    Pwflm /= (spd / nSampWin)

    # some power is lost when using hanning window
    Pwflm *= 8. / 3.

    # average over m
    Pwfl_mean = np.zeros((nWindow, int(nSampWin/2), ntrunc+1))
    for l in range(0, ntrunc+1):
        Pwfl_mean[:, :, l] = np.mean(Pwflm[:, :, l, ntrunc-l:ntrunc+l+1], axis=2)

    # sum over m
    Pwfl_sum = np.zeros((nWindow, int(nSampWin/2), ntrunc+1))
    for l in range(0, ntrunc+1):
        Pwfl_sum[:, :, l] = np.sum(Pwflm[:, :, l, ntrunc-l:ntrunc+l+1], axis=2)

    return Pwfl_mean, Pwfl_sum

In [ ]:
# analytic psd -- l
def psd_l_analytic(ff, ll):

    Pfl_analytic = np.zeros((int(nSampWin/2), ntrunc+1))

    for f in range(int(nSampWin/2)):
        for l in range(0, ntrunc+1):
            tau_l = tau_0 / (1 + lambda_0**2 * l * (l+1))
            epsilon_l = epsilon_0 * tau_l / tau_0
            Pfl_analytic[f, l] = 2 * epsilon_l**2 * tau_l / (1 + ff[f]**2 * tau_l**2 * 4 * np.pi**2)

    return Pfl_analytic

In [ ]:
# comp psd -- m
def psd_m(Fwflm):

    # compute power
    Pwflm = np.abs(Fwflm[:, :, :, :])**2

    # to get psd need to scale (divide) by frequency spacing = spd/(nSampWin)
    Pwflm /= (spd / nSampWin)

    # some power is lost when using hanning window
    Pwflm *= 8. / 3.

    # average over l
    Pwfm_mean = np.zeros((nWindow, int(nSampWin/2), 2*ntrunc+1))
    for m in range(-ntrunc, ntrunc+1):
        Pwfm_mean[:, :, ntrunc+m] = np.mean(Pwflm[:, :, np.abs(m):ntrunc+1, ntrunc+m], axis=2)

    # sum over l
    Pwfm_sum = np.zeros((nWindow, int(nSampWin/2), 2*ntrunc+1))
    for m in range(-ntrunc, ntrunc+1):
        Pwfm_sum[:, :, ntrunc+m] = np.sum(Pwflm[:, :, np.abs(m):ntrunc+1, ntrunc+m], axis=2)

    return Pwfm_mean, Pwfm_sum

In [ ]:
# analytic psd -- m
def psd_m_analytic(ff, ll, mm):

    Pflm_analytic = np.zeros((int(nSampWin/2), ntrunc+1, 2*ntrunc+1))

    for f in range(int(nSampWin/2)):
        for l in range(0, ntrunc+1):
            tau_l = tau_0 / (1 + lambda_0**2 * l * (l+1))
            epsilon_l = epsilon_0 * tau_l / tau_0
            Pflm_analytic[f, l, ntrunc-l:ntrunc+l+1] = 2 * epsilon_l**2 * tau_l / (1 + ff[f]**2 * tau_l**2 * 4 * np.pi**2)

    # average over l
    Pfm_analytic = np.zeros((int(nSampWin/2), 2*ntrunc+1))
    for m in range(-ntrunc, ntrunc+1):
        Pfm_analytic[:, ntrunc+m] = np.mean(Pflm_analytic[:, np.abs(m):ntrunc+1, ntrunc+m], axis=1)

    return Pfm_analytic

In [ ]:
# base dir
base_dir = (Path.cwd() / "../../").resolve()
data_dir = base_dir / "data"
save_dir = base_dir / "figures"

In [ ]:
file_name = "olr-2xdaily-1981-2010-space-time-analysis-window-360-skip-180.nc"
ds_observed = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# extract
Fwflm_observed = (ds_observed.Fwflm_real.values[:, 0:int(nSampWin/2), :, :] +
                  1j * ds_observed.Fwflm_imag.values[:, 0:int(nSampWin/2), :, :])

In [ ]:
# background realization
file_name = "ou-realization-2024-space-time-analysis-window-360-skip-180-epsilon0-5.8-lambda0-0.06-tau0-2.3.nc"
ds_ou = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# extract
Fwflm_ou = (ds_ou.Fwflm_real.values[:, 0:int(nSampWin/2), :, :] +
            1j * ds_ou.Fwflm_imag.values[:, 0:int(nSampWin/2), :, :])

In [ ]:
# precomputed p-values
file_name = "boot-statistics-spectral-space-realization-2024-epsilon0-5.8-lambda0-0.06-tau0-2.3.nc"
ds_boot = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# extract
p_value_l = ds_boot.p_value_l.values
p_value_m = ds_boot.p_value_m.values

In [ ]:
# psd -- l
Pwfl_observed, _ = psd_l(Fwflm_observed)
Pwfl_ou, _ = psd_l(Fwflm_ou)

Pfl_observed = np.mean(Pwfl_observed, axis=0)
Pfl_ou = np.mean(Pwfl_ou, axis=0)


In [ ]:
# psd -- m
Pwfm_observed, _ = psd_m(Fwflm_observed)
Pwfm_ou, _ = psd_m(Fwflm_ou)

Pfm_observed = np.mean(Pwfm_observed, axis=0)
Pfm_ou = np.mean(Pwfm_ou, axis=0)


In [ ]:
# psd -- analytic
Pfl_analytic = psd_l_analytic(ff, ll)
Pfm_analytic = psd_m_analytic(ff, ll, mm)


In [ ]:
# foreground
Bfl = Pfl_observed / Pfl_ou
Bfm = Pfm_observed / Pfm_ou


In [ ]:
fig_size = (16.00/2.54, 12.80/2.54)
fig = plt.figure(figsize=fig_size)

ax = []

ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.40, 08.15, 04.00, 04.00])))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [05.90, 08.15, 04.00, 04.00])))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [10.40, 08.15, 04.00, 04.00])))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.40, 02.80, 04.00, 04.00])))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [05.90, 02.80, 04.00, 04.00])))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [10.40, 02.80, 04.00, 04.00])))


##### ax0 #####

cs0 = ax[0].contourf(ll, ff, np.log10(Pfl_observed), levels=np.arange(-2.0, 3.1, 0.5), cmap='YlOrRd', extend='both')

ax[0].set_title(r'Observed', fontsize=10, fontweight='bold')
ax[0].set_xlabel(r'Multipole moment ($\ell$)', fontsize=10)
ax[0].set_ylabel(r'Frequency ($\omega$) [cpd]', fontsize=10)
ax[0].tick_params(axis='both', which='both', labelsize=9)
ax[0].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[0].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[0].set_xlim(0, None)
ax[0].set_ylim(1/360, 1)

secyax = ax[0].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelright=False, labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[0].text(00.05, 00.95, 'A', ha='left', va='top', transform=ax[0].transAxes, fontsize=10, fontweight='bold', color='black')


##### ax1 #####

cs1 = ax[1].contourf(ll, ff, np.log10(Pfl_ou), levels=cs0.levels, cmap='YlOrRd', extend='both')
cs1a = ax[1].contour(ll, ff, np.log10(Pfl_analytic), levels=cs0.levels, colors='black', linewidths=1, extend='both')

# ax[1].axvline(x=15, ymin=0, ymax=1, color='black', linestyle='--')

ax[1].set_title(r'Background ($\mathcal{E}$BCM)', fontsize=10, fontweight='bold')
ax[1].set_xlabel(r'Multipole moment ($\ell$)', fontsize=10)
ax[1].tick_params(axis='both', which='both', labelleft=False, labelsize=9)
ax[1].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[1].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[1].set_xlim(0, None)
ax[1].set_ylim(1/360, 1)

secyax = ax[1].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelright=False, labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[1].text(00.05, 00.95, 'B', ha='left', va='top', transform=ax[1].transAxes, fontsize=10, fontweight='bold', color='black')


##### ax2 #####

cs2 = ax[2].contourf(ll, ff, np.where(p_value_l < 0.001, np.log10(Bfl), np.nan), levels=np.linspace(-0.4, 0.4, 101), cmap=cmc, extend='both')
cs2 = ax[2].contourf(ll, ff, np.where(p_value_l < 0.001, np.log10(Bfl), np.nan), levels=np.linspace(-0.4, 0.4, 101), cmap=cmc, extend='both')

ax[2].add_patch(mpl.patches.Rectangle((9, 0.87), 20, 0.05, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))

ax[2].set_title(r'Foreground', fontsize=10, fontweight='bold')
ax[2].set_xlabel(r'Multipole moment ($\ell$)', fontsize=10)
ax[2].tick_params(axis='both', which='both', labelleft=False, labelsize=9)
ax[2].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[2].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[2].set_xlim(0, None)
ax[2].set_ylim(1/360, 1)

secyax = ax[2].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_ylabel('Period [day]', fontsize=10)
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelright=True, labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[2].text(00.05, 00.95, 'C', ha='left', va='top', transform=ax[2].transAxes, fontsize=10, fontweight='bold', color='black')


##### ax3 #####

cs3 = ax[3].contourf(mm, ff, np.log10(Pfm_observed[:, ::-1]), levels=cs0.levels, cmap='YlOrRd', extend='both')

ax[3].set_xlabel(r'Wavenumber ($m$)', fontsize=10)
ax[3].set_ylabel(r'Frequency ($\omega$) [cpd]', fontsize=10)
ax[3].tick_params(axis='both', which='both', labelsize=9)
ax[3].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[3].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[3].set_xticks(np.array([-60, -30, 0, 30, 60]))
ax[3].set_xlim(-72, 72)
ax[3].set_ylim(1/360, 1)

secyax = ax[3].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelright=False, labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[3].text(00.05, 00.95, 'D', ha='left', va='top', transform=ax[3].transAxes, fontsize=10, fontweight='bold', color='black')


##### ax4 #####

cs4 = ax[4].contourf(mm, ff, np.log10(Pfm_ou[:, ::-1]), levels=cs0.levels, cmap='YlOrRd', extend='both')
cs4a = ax[4].contour(mm, ff, np.log10(Pfm_analytic), levels=cs0.levels, colors='black', linewidths=1, extend='both')

ax[4].set_xlabel(r'Wavenumber ($m$)', fontsize=10)
ax[4].tick_params(axis='both', which='both', labelleft=False, labelsize=9)
ax[4].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[4].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[4].set_xticks(np.array([-60, -30, 0, 30, 60]))
ax[4].set_xlim(-72, 72)
ax[4].set_ylim(1/360, 1)

secyax = ax[4].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelright=False, labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[4].text(00.05, 00.95, 'E', ha='left', va='top', transform=ax[4].transAxes, fontsize=10, fontweight='bold', color='black')

cax = fig.add_axes(ax_pos_cm_to_absolute(fig_size, [02.00, 01.10, 07.30, 00.25])) 
cbar = fig.colorbar(cs3, cax=cax, orientation='horizontal', extend='neither')
cax.set_xlabel(r'$\log_{10}$(Power) [W m$^{-2}$]$^{2}$ day')
cax.tick_params(axis='both', which='both', labelsize=9)


##### ax5 #####

cs5 = ax[5].contourf(mm, ff, np.where(p_value_m < 0.001, np.log10(Bfm[:, ::-1]), np.nan), levels=np.linspace(-0.4, 0.4, 101), cmap=cmc, extend='both')
cs5 = ax[5].contourf(mm, ff, np.where(p_value_m < 0.001, np.log10(Bfm[:, ::-1]), np.nan), levels=np.linspace(-0.4, 0.4, 101), cmap=cmc, extend='both')


ax[5].plot(mm[ntrunc:ntrunc+18+1], (10*10)**0.5 * mm[ntrunc:ntrunc+18+1] * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[5].plot(mm[ntrunc:ntrunc+9+1], (100*10)**0.5 * mm[ntrunc:ntrunc+9+1] * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
intercept = (100*10)**0.5 * 9
slope = ((10*10)**0.5 * 18 - (100*10)**0.5 * 9) / (18 - 9)
ax[5].plot(mm[ntrunc+9:ntrunc+18+1], (intercept + slope * (mm[ntrunc+9:ntrunc+18+1] - 9))* 86400 / 4e7, color='black', linestyle='-', linewidth=1)

ax[5].plot(mm[ntrunc-18:ntrunc+1], -(10*10)**0.5 * mm[ntrunc-18:ntrunc+1] * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[5].plot(mm[ntrunc-9:ntrunc+1], -(100*10)**0.5 * mm[ntrunc-9:ntrunc+1] * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
intercept = -(100*10)**0.5 * (-9)
slope = (-(10*10)**0.5 * (-18) + (100*10)**0.5 * (-9)) / (-18 + 9)
ax[5].plot(mm[ntrunc-9:ntrunc-18-1:-1], (intercept + slope * (mm[ntrunc-9:ntrunc-18-1:-1] + 9))* 86400 / 4e7, color='black', linestyle='-', linewidth=1)


ax[5].add_patch(mpl.patches.Rectangle((11.5, 0.08), 5, 0.06, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))
ax[5].add_patch(mpl.patches.Rectangle((-26, 0.87), 20, 0.05, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))
ax[5].add_patch(mpl.patches.Rectangle((-17, 0.76), 5, 0.05, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))
ax[5].add_patch(mpl.patches.Rectangle((-32, 0.76), 5, 0.05, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))


ax[5].set_xlabel(r'Wavenumber ($m$)', fontsize=10)
ax[5].tick_params(axis='both', which='both', labelleft=False, labelsize=9)
ax[5].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[5].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[5].set_xticks(np.array([-60, -30, 0, 30, 60]))
ax[5].set_xlim(-72, 72)
ax[5].set_ylim(1/360, 1)

secyax = ax[5].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_ylabel('Period [day]', fontsize=10)
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[5].text(00.05, 00.95, 'F', ha='left', va='top', transform=ax[5].transAxes, fontsize=10, fontweight='bold', color='black')


cax = fig.add_axes(ax_pos_cm_to_absolute(fig_size, [10.80, 01.10, 03.20, 00.25])) 
cbar = fig.colorbar(cs5, cax=cax, orientation='horizontal', extend='neither') #, label=r'$\mathrm{W m^{-2}}$')
cax.set_xlabel(r'$\log_{10}$(Ratio)', fontsize=10)
cax.set_xticks(np.array([-0.3, 0.0, 0.3])) #, labels=['-0.6', '-0.3', '0.0', '+0.3', '+0.6'])
cax.tick_params(axis='both', which='both', labelsize=9)

for i in range(6):
    ax[i].grid()


In [ ]:
file_name = "fig-04"
Path(save_dir).mkdir(parents=True, exist_ok=True)
fig.savefig(str(save_dir / file_name) + ".png", dpi=600, format='png', facecolor='white')
fig.savefig(str(save_dir / file_name) + ".pdf", dpi=600, format='pdf', facecolor='white')

In [ ]:
# squeezing a little bit

In [ ]:
fig_size = (14.20/2.54, 11.60/2.54)
fig = plt.figure(figsize=fig_size)

ax = []

ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.30, 07.50, 03.50, 03.50])))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [05.30, 07.50, 03.50, 03.50])))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [09.30, 07.50, 03.50, 03.50])))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.30, 02.70, 03.50, 03.50])))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [05.30, 02.70, 03.50, 03.50])))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [09.30, 02.70, 03.50, 03.50])))


##### ax0 #####

cs0 = ax[0].contourf(ll, ff, np.log10(Pfl_observed), levels=np.arange(-2.0, 3.1, 0.5), cmap='YlOrRd', extend='both')

ax[0].set_title(r'Observed', fontsize=10, fontweight='bold')
ax[0].set_xlabel(r'Multipole moment ($\ell$)', fontsize=10)
ax[0].set_ylabel(r'Frequency ($\omega$) [cpd]', fontsize=10)
ax[0].tick_params(axis='both', which='both', labelsize=9)
ax[0].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[0].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[0].set_xlim(0, None)
ax[0].set_ylim(1/360, 1)

secyax = ax[0].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelright=False, labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[0].text(00.05, 00.95, 'A', ha='left', va='top', transform=ax[0].transAxes, fontsize=10, fontweight='bold', color='black')


##### ax1 #####

cs1 = ax[1].contourf(ll, ff, np.log10(Pfl_ou), levels=cs0.levels, cmap='YlOrRd', extend='both')
cs1a = ax[1].contour(ll, ff, np.log10(Pfl_analytic), levels=cs0.levels, colors='black', linewidths=1, extend='both')

# ax[1].axvline(x=15, ymin=0, ymax=1, color='black', linestyle='--')

ax[1].set_title(r'Background ($\mathcal{E}$BCM)', fontsize=10, fontweight='bold')
ax[1].set_xlabel(r'Multipole moment ($\ell$)', fontsize=10)
ax[1].tick_params(axis='both', which='both', labelleft=False, labelsize=9)
ax[1].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[1].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[1].set_xlim(0, None)
ax[1].set_ylim(1/360, 1)

secyax = ax[1].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelright=False, labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[1].text(00.05, 00.95, 'B', ha='left', va='top', transform=ax[1].transAxes, fontsize=10, fontweight='bold', color='black')


##### ax2 #####

cs2 = ax[2].contourf(ll, ff, np.where(p_value_l < 0.001, np.log10(Bfl), np.nan), levels=np.linspace(-0.4, 0.4, 101), cmap=cmc, extend='both')
cs2 = ax[2].contourf(ll, ff, np.where(p_value_l < 0.001, np.log10(Bfl), np.nan), levels=np.linspace(-0.4, 0.4, 101), cmap=cmc, extend='both')

ax[2].add_patch(mpl.patches.Rectangle((9, 0.87), 20, 0.05, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))

ax[2].set_title(r'Foreground', fontsize=10, fontweight='bold')
ax[2].set_xlabel(r'Multipole moment ($\ell$)', fontsize=10)
ax[2].tick_params(axis='both', which='both', labelleft=False, labelsize=9)
ax[2].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[2].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[2].set_xlim(0, None)
ax[2].set_ylim(1/360, 1)

secyax = ax[2].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_ylabel('Period [day]', fontsize=10)
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelright=True, labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[2].text(00.05, 00.95, 'C', ha='left', va='top', transform=ax[2].transAxes, fontsize=10, fontweight='bold', color='black')


##### ax3 #####

cs3 = ax[3].contourf(mm, ff, np.log10(Pfm_observed[:, ::-1]), levels=cs0.levels, cmap='YlOrRd', extend='both')

ax[3].set_xlabel(r'Wavenumber ($m$)', fontsize=10)
ax[3].set_ylabel(r'Frequency ($\omega$) [cpd]', fontsize=10)
ax[3].tick_params(axis='both', which='both', labelsize=9)
ax[3].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[3].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[3].set_xticks(np.array([-60, -30, 0, 30, 60]))
ax[3].set_xlim(-72, 72)
ax[3].set_ylim(1/360, 1)

secyax = ax[3].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelright=False, labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[3].text(00.05, 00.95, 'D', ha='left', va='top', transform=ax[3].transAxes, fontsize=10, fontweight='bold', color='black')


##### ax4 #####

cs4 = ax[4].contourf(mm, ff, np.log10(Pfm_ou[:, ::-1]), levels=cs0.levels, cmap='YlOrRd', extend='both')
cs4a = ax[4].contour(mm, ff, np.log10(Pfm_analytic), levels=cs0.levels, colors='black', linewidths=1, extend='both')

ax[4].set_xlabel(r'Wavenumber ($m$)', fontsize=10)
ax[4].tick_params(axis='both', which='both', labelleft=False, labelsize=9)
ax[4].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[4].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[4].set_xticks(np.array([-60, -30, 0, 30, 60]))
ax[4].set_xlim(-72, 72)
ax[4].set_ylim(1/360, 1)

secyax = ax[4].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelright=False, labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[4].text(00.05, 00.95, 'E', ha='left', va='top', transform=ax[4].transAxes, fontsize=10, fontweight='bold', color='black')

cax = fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.80, 01.10, 06.50, 00.25])) 
cbar = fig.colorbar(cs3, cax=cax, orientation='horizontal', extend='neither')
cax.set_xlabel(r'$\log_{10}$(Power) [W m$^{-2}$]$^{2}$ day')
cax.tick_params(axis='both', which='both', labelsize=9)


##### ax5 #####

cs5 = ax[5].contourf(mm, ff, np.where(p_value_m < 0.001, np.log10(Bfm[:, ::-1]), np.nan), levels=np.linspace(-0.4, 0.4, 101), cmap=cmc, extend='both')
cs5 = ax[5].contourf(mm, ff, np.where(p_value_m < 0.001, np.log10(Bfm[:, ::-1]), np.nan), levels=np.linspace(-0.4, 0.4, 101), cmap=cmc, extend='both')


ax[5].plot(mm[ntrunc:ntrunc+18+1], (10*10)**0.5 * mm[ntrunc:ntrunc+18+1] * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[5].plot(mm[ntrunc:ntrunc+9+1], (100*10)**0.5 * mm[ntrunc:ntrunc+9+1] * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
intercept = (100*10)**0.5 * 9
slope = ((10*10)**0.5 * 18 - (100*10)**0.5 * 9) / (18 - 9)
ax[5].plot(mm[ntrunc+9:ntrunc+18+1], (intercept + slope * (mm[ntrunc+9:ntrunc+18+1] - 9))* 86400 / 4e7, color='black', linestyle='-', linewidth=1)

ax[5].plot(mm[ntrunc-18:ntrunc+1], -(10*10)**0.5 * mm[ntrunc-18:ntrunc+1] * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
ax[5].plot(mm[ntrunc-9:ntrunc+1], -(100*10)**0.5 * mm[ntrunc-9:ntrunc+1] * 86400 / 4e7, color='black', linestyle='-', linewidth=1)
intercept = -(100*10)**0.5 * (-9)
slope = (-(10*10)**0.5 * (-18) + (100*10)**0.5 * (-9)) / (-18 + 9)
ax[5].plot(mm[ntrunc-9:ntrunc-18-1:-1], (intercept + slope * (mm[ntrunc-9:ntrunc-18-1:-1] + 9))* 86400 / 4e7, color='black', linestyle='-', linewidth=1)


ax[5].add_patch(mpl.patches.Rectangle((11.5, 0.08), 5, 0.06, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))
ax[5].add_patch(mpl.patches.Rectangle((-26, 0.87), 20, 0.05, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))
ax[5].add_patch(mpl.patches.Rectangle((-17, 0.76), 5, 0.05, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))
ax[5].add_patch(mpl.patches.Rectangle((-32, 0.76), 5, 0.05, edgecolor = 'black', facecolor = 'none', fill=False, lw=1))


ax[5].set_xlabel(r'Wavenumber ($m$)', fontsize=10)
ax[5].tick_params(axis='both', which='both', labelleft=False, labelsize=9)
ax[5].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[5].tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')
ax[5].set_xticks(np.array([-60, -30, 0, 30, 60]))
ax[5].set_xlim(-72, 72)
ax[5].set_ylim(1/360, 1)

secyax = ax[5].secondary_yaxis('right', functions=(freq_to_time, freq_to_time))
secyax.set_ylabel('Period [day]', fontsize=10)
secyax.set_yticks(np.array([1/0.2, 1/0.4, 1/0.6, 1/0.8, 1]))
secyax.yaxis.set_major_formatter(StrMethodFormatter(u"{x:.2f}"))
secyax.tick_params(axis='both', which='both', labelsize=9)
secyax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secyax.tick_params(axis='both', which='minor', direction='out', size=2.0, color='0.8')

ax[5].text(00.05, 00.95, 'F', ha='left', va='top', transform=ax[5].transAxes, fontsize=10, fontweight='bold', color='black')


cax = fig.add_axes(ax_pos_cm_to_absolute(fig_size, [09.55, 01.10, 03.00, 00.25])) 
cbar = fig.colorbar(cs5, cax=cax, orientation='horizontal', extend='neither') #, label=r'$\mathrm{W m^{-2}}$')
cax.set_xlabel(r'$\log_{10}$(Ratio)', fontsize=10)
cax.set_xticks(np.array([-0.3, 0.0, 0.3])) #, labels=['-0.6', '-0.3', '0.0', '+0.3', '+0.6'])
cax.tick_params(axis='both', which='both', labelsize=9)

for i in range(6):
    ax[i].grid()


In [ ]:
file_name = "fig-04"
Path(save_dir).mkdir(parents=True, exist_ok=True)
fig.savefig(str(save_dir / file_name) + ".png", dpi=600, format='png', facecolor='white')
fig.savefig(str(save_dir / file_name) + ".pdf", dpi=600, format='pdf', facecolor='white')